# MODELOS

IMPORTACION DE LIBRERIAS

In [2]:
import joblib
from collections import Counter

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical


CARGA DE DATOS

In [3]:
df = pd.read_csv(r'.\data\observations_full.csv')

df['date'] = pd.to_datetime(df['date'])

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

SVM

In [4]:
# SELECCIONAR FEATURES Y TARGET
features = ['year', 'month', 'day', 'precipitation', 'wind', 'humidity', 'estacion_id']
target = 'weather_id'

# DEFINIR X Y y
X = df[features]
y = df[target]

# BALANCEAR LAS CLASES CON SMOTE
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

# SEPARAR ENTRE TRAIN Y TEST
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)

# ESCALAR LOS DATOS
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# CARGAR EL MODELO Y ENTRENARLO
model = SVC(probability=True, C=0.1, gamma='scale', kernel='rbf', class_weight='balanced')
model.fit(X_train_scaled, y_train)

# REALIZAR PREDICCIONES
y_pred = model.predict(X_test_scaled)

# CALCULAR ACCURACY
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

# REPORTE DE CLASIFICACION
print("Classification Report:")
print(classification_report(y_test, y_pred))

# MATRIZ DE CONFUSION
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ROC AUC PARA DETERMIANAR DESEMPEÑO DEL MODELO POR CLASES
roc_auc = roc_auc_score(y_test, model.predict_proba(X_test_scaled), multi_class='ovr')
print(f"ROC AUC: {roc_auc}")

# GUARDAR EL MODELO USANDO JOBLIB
# COMENTADO PARA EVITAR GUARDAR MODELOS NUEVOS
#joblib.dump(model, r'.\main\Modelos\SVM_weather_id.pkl')

Accuracy: 0.7854054624064115
Classification Report:
              precision    recall  f1-score   support

           1       0.82      0.86      0.84      1897
           2       0.79      0.82      0.81      1896
           3       0.75      0.44      0.56      1897
           4       0.68      0.84      0.75      1897
           5       0.89      0.96      0.92      1896

    accuracy                           0.79      9483
   macro avg       0.79      0.79      0.78      9483
weighted avg       0.79      0.79      0.78      9483

Confusion Matrix:
[[1631    2  116  148    0]
 [  15 1557  150  174    0]
 [ 255  263  835  360  184]
 [  96  149    6 1602   44]
 [   0    0    3   70 1823]]
ROC AUC: 0.9429371569285117


XGBoosting Classifier

In [6]:
# SELECCIONAR FEATURES Y TARGET
features = ['year', 'month', 'day', 'precipitation', 'wind', 'humidity', 'estacion_id']
target = 'weather_id'

# DEFINIR X Y y
X = df[features]
y = df[target]

# AJUSTAR LAS ETIQUETAS PARA QUE EMPIECEN EN 0 (REQUERIMIENTO DE XGBOOST)
y_adjusted = y - 1

# DISTRIBUCION ORIGINAL DE LAS VARIABLES
print("Distribución original de las clases:", Counter(y_adjusted))

# REALIZAR UN SMOTE PARA BALANCEAR LAS CLASES
smote = SMOTE(random_state=42, k_neighbors=10)
X_resampled, y_resampled = smote.fit_resample(X, y_adjusted)

# NUEVA DISTRIBUCION DE LAS CLASES
print("Distribución de clases después de SMOTE:", Counter(y_resampled))

# SEPARAR ENTRE TRAIN Y TEST
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# CREAR EL MODELO XGBOOST
model = XGBClassifier(random_state=42, n_estimators=300, max_depth=30)

# ENTRENAR EL MODELO
model.fit(X_train, y_train)

# REALIZAR PREDICCIONES
y_pred = model.predict(X_test)

# RECONVERTIR LAS PREDICCIONES Y LAS ETIQUETAS A LOS VALORES ORIGINALES
y_pred = y_pred + 1
y_test = y_test + 1

# CALCULAR ACCURACY
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

# REPORTE DE CLASIFICACION
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))

# MATRIZ DE CONFUSION
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

# GUARDAR EL MODELO USANDO JOBLIB
# COMENTADO PARA EVITAR GUARDAR MODELOS NUEVOS
#joblib.dump(model, r'..\main\Modelos\XGB_weather_id.pkl')

Distribución original de las clases: Counter({1: 9483, 0: 9024, 2: 5959, 3: 429, 4: 105})
Distribución de clases después de SMOTE: Counter({0: 9483, 2: 9483, 1: 9483, 3: 9483, 4: 9483})
Accuracy: 0.9244964673626489

Reporte de clasificación:
              precision    recall  f1-score   support

           1       0.89      0.97      0.93      1897
           2       0.89      0.96      0.92      1910
           3       0.93      0.71      0.80      1861
           4       0.93      0.98      0.95      1892
           5       0.99      1.00      0.99      1923

    accuracy                           0.92      9483
   macro avg       0.93      0.92      0.92      9483
weighted avg       0.93      0.92      0.92      9483


Matriz de confusión:
[[1847    1   29   20    0]
 [   1 1825   58   26    0]
 [ 203  215 1320   97   26]
 [  13   14   12 1852    1]
 [   0    0    0    0 1923]]


['.\\..\\Modelos\\XGB_weather_id.pkl']

RANDOM FOREST CLASSIFIER

In [12]:
# SELECCIONAR FEATURES Y TARGET
features = ['year', 'month', 'day', 'precipitation', 'wind', 'humidity', 'estacion_id']
target = 'weather_id'

# DEFINIR X Y y
X = df[features]
y = df[target]

# AJUSTAR LAS ETIQUETAS PARA QUE EMPIECEN EN 0 (REQUERIMIENTO DE RANDOM FOREST)
y_adjusted = y - 1

# DISTRIBUCION ORIGINAL DE LAS VARIABLES
print("Distribución original de las clases:", Counter(y_adjusted))

# REALIZAR UN SMOTE PARA BALANCEAR LAS CLASES
smote = SMOTE(random_state=42,k_neighbors=10)
X_resampled, y_resampled = smote.fit_resample(X, y_adjusted)

# NUEVA DISTRIBUCION DE LAS CLASES
print("Distribución de clases después de SMOTE:", Counter(y_resampled))

# SEPARAR ENTRE TRAIN Y TEST
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# CREAR EL MODELO RANDOM FOREST CLASSIFIER
model = RandomForestClassifier(random_state=42, n_estimators=300, max_depth=30)

# ENTRENAR EL MODELO
model.fit(X_train, y_train)

# REALIZAR PREDICCIONES
y_pred = model.predict(X_test)

# RECONVERTIR LAS PREDICCIONES Y LAS ETIQUETAS A LOS VALORES ORIGINALES
y_pred = y_pred + 1
y_test = y_test + 1

# CALCULAR ACCURACY
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

# REPORTE DE CLASIFICACION
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))

# MATRIZ DE CONFUSION
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

# GUARDAR EL MODELO USANDO JOBLIB
# COMENTADO PARA EVITAR GUARDAR MODELOS NUEVOS
#joblib.dump(model, r'.\main\Modelos\RFC_weather_id.pkl')

Distribución original de las clases: Counter({1: 9483, 0: 9024, 2: 5959, 3: 429, 4: 105})
Distribución de clases después de SMOTE: Counter({0: 9483, 2: 9483, 1: 9483, 3: 9483, 4: 9483})
Accuracy: 0.9128967626278603

Reporte de clasificación:
              precision    recall  f1-score   support

           1       0.87      0.99      0.92      1897
           2       0.87      0.98      0.92      1910
           3       0.97      0.62      0.76      1861
           4       0.90      0.98      0.94      1892
           5       0.98      1.00      0.99      1923

    accuracy                           0.91      9483
   macro avg       0.92      0.91      0.91      9483
weighted avg       0.92      0.91      0.91      9483


Matriz de confusión:
[[1870    0    9   18    0]
 [   0 1864   17   29    0]
 [ 265  253 1155  156   32]
 [  18   19    9 1846    0]
 [   0    0    1    0 1922]]


RNN

In [4]:
# SELECCIONAR FEATURES Y TARGET
features = ['year', 'month', 'day', 'precipitation', 'wind', 'humidity', 'estacion_id']
target = 'weather_id'

# DEFINIR X Y y
X = df[features]
y = df[target]

# ESCALAR Y BALANCEAR LAS CLASES
scaler = StandardScaler()
X_resampled, y_resampled = SMOTE(random_state=42).fit_resample(X, y)
X_scaled = scaler.fit_transform(X_resampled)

# REALIZAR ONE HOT AL TARGET PARA VERIFICAR QUE NO ES CATEGORICA
y_encoded = to_categorical(y_resampled)

# SEPARAR ENTRE TRAIN Y TEST
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

# ======== CREAR LA RED NEURONAL ========
model = Sequential()
# PRIEMRA CAPA CON MAS NEURONAS PARA MAYOR CAPACIDAD DE APRENDIZAJE
model.add(Dense(256, input_dim=X_train.shape[1], activation='relu'))  
model.add(Dropout(0.3))  # REGULARIZACION PARA EVITAR OVERFITTING

# SEGUNDA CAPA OCULTA
model.add(Dense(128, activation='relu'))  
model.add(Dropout(0.3))  # REGULARIZACION PARA EVITAR OVERFITTING

# TERCERA CAPA OCULTA
model.add(Dense(64, activation='relu'))  
model.add(Dense(y_encoded.shape[1], activation='softmax'))  # CAPA DE SALIDA PARA CLASIFICACION MULTICLASE
# =====================================

# COMPLIAR EL MODELO
model.compile(optimizer='adam', 
              loss='categorical_crossentropy',  # FUNCION DE PERDIDA PARA CLASIFICACION MULTICLASE
              metrics=['accuracy'])

# ENTRENAR EL MODELO
history = model.fit(X_train, y_train, 
                    validation_data=(X_test, y_test), 
                    epochs=20,  # EPOCAS DEL MODELO
                    batch_size=64,  # TAMAÑO DEL LOTE
                    verbose=2)

# EVALUAR EL MODELO
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=2)
print(f'\nPérdida en prueba: {test_loss}')
print(f'Precisión en prueba: {test_accuracy}')

# PREDECIR RESULTADOS
predictions = model.predict(X_test)

# DESCODIFICAR LOS RESULTADOS SU CLASE ORIGINAL
predicted_classes = predictions.argmax(axis=1)

# DESCODIFICAR LA CLASE ORIGINAL
y_test_classes = y_test.argmax(axis=1)

# MATRIZ DE CONFUSION
cm = confusion_matrix(y_test_classes, predicted_classes)
print("\nMatriz de Confusión:")
print(cm)

# REPORTE DE CLASIFICACION
print("\nReporte de Clasificación:")
print(classification_report(y_test_classes, predicted_classes))

# GUARDAR EL MODELO USANDO EL PROPIO SISTEMA DE GUARDADO DE tensorflow
# COMENTADO PARA EVITAR GUARDAR MODELOS NUEVOS
# model.save(r'.\main\Modelos\RNN_weather_id.h5')


Epoch 1/20


c:\Users\jpetit.sta\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


593/593 - 3s - 4ms/step - accuracy: 0.7030 - loss: 0.7364 - val_accuracy: 0.7918 - val_loss: 0.5283
Epoch 2/20
593/593 - 1s - 2ms/step - accuracy: 0.7672 - loss: 0.5760 - val_accuracy: 0.8102 - val_loss: 0.4903
Epoch 3/20
593/593 - 1s - 2ms/step - accuracy: 0.7862 - loss: 0.5346 - val_accuracy: 0.8200 - val_loss: 0.4604
Epoch 4/20
593/593 - 1s - 2ms/step - accuracy: 0.7978 - loss: 0.5067 - val_accuracy: 0.8188 - val_loss: 0.4543
Epoch 5/20
593/593 - 1s - 2ms/step - accuracy: 0.8065 - loss: 0.4837 - val_accuracy: 0.8372 - val_loss: 0.4233
Epoch 6/20
593/593 - 1s - 2ms/step - accuracy: 0.8166 - loss: 0.4664 - val_accuracy: 0.8434 - val_loss: 0.4032
Epoch 7/20
593/593 - 1s - 2ms/step - accuracy: 0.8207 - loss: 0.4517 - val_accuracy: 0.8442 - val_loss: 0.4038
Epoch 8/20
593/593 - 1s - 2ms/step - accuracy: 0.8291 - loss: 0.4374 - val_accuracy: 0.8441 - val_loss: 0.3911
Epoch 9/20
593/593 - 1s - 2ms/step - accuracy: 0.8320 - loss: 0.4276 - val_accuracy: 0.8492 - val_loss: 0.3860
Epoch 10/20



[[1791    1    7   98    0]
 [  16 1800    6   88    0]
 [ 333  293  884  292   59]
 [  18   37    0 1825   12]
 [   0    0    2    0 1921]]

Reporte de Clasificación:
              precision    recall  f1-score   support

           1       0.83      0.94      0.88      1897
           2       0.84      0.94      0.89      1910
           3       0.98      0.48      0.64      1861
           4       0.79      0.96      0.87      1892
           5       0.96      1.00      0.98      1923

    accuracy                           0.87      9483
   macro avg       0.88      0.87      0.85      9483
weighted avg       0.88      0.87      0.85      9483

